#BIO 라벨링

{
  "tokens": ["너랑", "같이", "자", "고", "싶", "어"],
  "labels": ["O", "O", "B-RISK", "I-RISK", "I-RISK", "I-RISK"]
}

라벨은 보통 다음 중 하나:

- O (기타)

- B-RISK (위험 단어 시작)

- I-RISK (위험 단어 내부)

## 전체 데이터 형식

```
data = [
    {
        "tokens": ["사진만", "하나", "보내줘", "."],
        "labels": [1, 2, 2, 0],  # B-RISK, I-RISK, I-RISK, O
    },
    {
        "tokens": ["어른들이", "몰라도", "돼", "."],
        "labels": [1, 2, 2, 0],
    },
]
```

## 처리 전략

- 위험 단어 사전 포함 -> source = "keyword"

- 미포함 + 의미 기반 위험 -> source = "semantic"

### Step 1. datasets.Dataset 생성

In [7]:
from datasets import Dataset
import json

# JSON 파일 불러오기
with open("sample_token_label_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Hugging Face Dataset으로 변환
dataset = Dataset.from_list(data)

### Step 2. 토크나이저 및 모델 준비

In [11]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3)  # O, B, I

# label mapping 추가
label_list = ["O", "B", "I"]
model.config.id2label = {i: l for i, l in enumerate(label_list)}
model.config.label2id = {l: i for i, l in enumerate(label_list)}

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Step 3. 전처리 함수 (Tokenize + Label 정렬)

In [13]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_offsets_mapping=True
    )

    all_labels = []
    for i in range(len(examples["labels"])):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Only label the first token of a given word.
                label_ids.append(examples["labels"][i][word_idx])
            else:
                label_ids.append(-100) # Set subsequent subword tokens to -100
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized

### Step 4. 데이터 전처리 적용

In [14]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

### Step 5. 학습 설정

In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evalu_strategy="epoch",
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="epoch"
)

### Step 6. Trainer로 학습

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

trainer.train()


<ipython-input-17-3241670546>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: a01090314511 (a01090314511-dong-eui-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,No log,0.701374
2,No log,0.501762
3,No log,0.423502


TrainOutput(global_step=3, training_loss=0.7574721972147623, metrics={'train_runtime': 125.7427, 'train_samples_per_second': 0.095, 'train_steps_per_second': 0.024, 'total_flos': 195974339328.0, 'train_loss': 0.7574721972147623, 'epoch': 3.0})

### Step 7. 모델 저장

In [18]:
# 모델 저장
trainer.save_model("./my-model")  # 원하는 경로에 저장
tokenizer.save_pretrained("./my-model")  # 토크나이저도 함께 저장

('./my-model/tokenizer_config.json',
 './my-model/special_tokens_map.json',
 './my-model/vocab.txt',
 './my-model/added_tokens.json',
 './my-model/tokenizer.json')

### Step 8. 예측 테스트

In [27]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

text = "잠옷 입고있어?"
inputs = tokenizer(text.split(), return_tensors="pt", is_split_into_words=True)
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
pred_ids = predictions[0].tolist()

for token, pred in zip(tokens, pred_ids):
    print(f"{token}: {pred}")

# 문장 전체 위험 판별
is_risky = any(pred in [1, 2] for pred in pred_ids)
print("\n⚠️ 위험한 문장입니다" if is_risky else "\n✅ 안전한 문장입니다")

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[CLS]: 0
잠옷: 1
입: 0
##고: 0
##있: 0
##어: 0
?: 0
[SEP]: 0

⚠️ 위험한 문장입니다


#### 테스트 추가

아래와 같이 위험 토큰만 출력도 가능

In [ ]:
risk_label_map = {1: "B-RISK", 2: "I-RISK"}

for token, pred in zip(tokens, pred_ids):
    if pred in risk_label_map:
        print(f"{token}: {risk_label_map[pred]}")